# Lasso 모델 검증: 신검단중앙역 금강펜테리움센트럴파크

## 1. 라이브러리 임포트

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso

# 한글 폰트 설정
matplotlib.rc('font', family='Malgun Gothic')
matplotlib.rc('axes', unicode_minus=False)

## 2. 데이터 불러오기

In [ ]:
# 훈련용 데이터 (기존 신도시) 및 테스트용 데이터 (검단) 경로 설정
train_folder = r"C:\Users\SJ\Desktop\26-1\데이터마이닝\파주시아파트"
test_folder = r"C:\Users\SJ\Desktop\26-1\데이터마이닝\검단아파트"

# 기존 신도시 데이터에서 광교, 판교, 운정만 훈련 데이터로 사용
df_all = pd.read_csv(train_folder + r"\new_city2.csv", encoding='utf-8-sig')
df_train = df_all[df_all['도시명'].isin(['광교', '판교', '운정'])].copy()

# 검단 데이터 불러오기
df_test = pd.read_csv(test_folder + r"\geomdan_merged.csv", encoding='utf-8-sig')

## 3. 데이터 전처리

In [ ]:
# 거래금액 쉼표 제거 후 정수 변환
df_train['거래금액(만원)'] = df_train['거래금액(만원)'].str.replace(',', '').astype(int)
df_test['거래금액(만원)'] = df_test['거래금액(만원)'].str.replace(',', '').astype(int)

# 전용면적당 가격(m2당가격) 파생변수 생성
df_train['m2당가격'] = df_train['거래금액(만원)'] / df_train['전용면적(㎡)']
df_test['m2당가격'] = df_test['거래금액(만원)'] / df_test['전용면적(㎡)']

## 4. 훈련 데이터 구성

In [ ]:
# 발표후경과년수 3년 미만 데이터 제거 (신도시 초기 불안정 거래 제외)
df_train = df_train[df_train['발표후경과년수'] >= 3]
df_test = df_test[df_test['발표후경과년수'] >= 3]

# 2022년 이하만 훈련에 사용
df_train = df_train[df_train['계약연도'] <= 2022]
df_geomdan_train = df_test[df_test['계약연도'] <= 2022].copy()
df_train = pd.concat([df_train, df_geomdan_train], ignore_index=True)

## 5. 이상치 제거

In [ ]:
# 전용면적 33제곱미터 미만 제거
df_train = df_train[df_train['전용면적(㎡)'] >= 33]

# m2당가격 z-score 기준 이상치 제거 (|z| > 2인 데이터 제거)
mean = df_train['m2당가격'].mean()
std = df_train['m2당가격'].std()
z_scores = (df_train['m2당가격'] - mean) / std
df_train = df_train[z_scores.abs() <= 2]

print(f"훈련 데이터: {len(df_train)}건 (~2022년)")

## 6. 독립변수 설정 및 표준화

In [ ]:
# 독립변수 목록
features = ['건축년도', '층',
            '지하철호선개수', '기차역까지의거리',
            '가장 가까운 지하철역까지의 거리', '가장 가까운 IC와의 거리',
            '발표후경과년수', 'CPI', '계약연도', '서울도심거리',
            '단지별_세대수', '도시별_세대수']

# 결측치 제거
df_train = df_train.dropna(subset=features + ['m2당가격'])

# 훈련 데이터 독립변수와 타겟 분리
train_input = df_train[features]
train_target = df_train['m2당가격']

# 표준화
ss = StandardScaler()
ss.fit(train_input)
train_scaled = ss.transform(train_input)

## 7. Lasso 모델 학습

In [ ]:
# 최적 alpha=0.001로 Lasso 모델 학습
lasso_best = Lasso(alpha=0.001, max_iter=100000)
lasso_best.fit(train_scaled, train_target)

print(f"훈련 R²: {lasso_best.score(train_scaled, train_target):.4f}")

## 8. 모델 검증: 신검단중앙역 금강펜테리움센트럴파크 (2026년 실제 vs 예측)

In [ ]:
# 금강펜테리움센트럴파크 기본 특성값
금강_기본 = {
    '건축년도': 2025,
    '층': 12,
    '지하철호선개수': 3,
    '기차역까지의거리': 16.7135,
    '가장 가까운 지하철역까지의 거리': 0.894,
    '가장 가까운 IC와의 거리': 5.804,
    '발표후경과년수': 20,
    'CPI': 116.61,
    '계약연도': 2026,
    '서울도심거리': 25.136,
    '단지별_세대수': 1049.0,
    '도시별_세대수': 60833.0,
}

In [ ]:
# 2026년 평균 전용면적 및 실제 거래금액 (38건 평균)
면적_검증 = 86.23
실제_거래금액 = 52093

# 모델 예측
입력 = pd.DataFrame([금강_기본])[features]
입력_scaled = ss.transform(입력)
예측_m2 = lasso_best.predict(입력_scaled)[0]
예측_거래금액 = int(예측_m2 * 면적_검증)
오차율 = abs(예측_거래금액 - 실제_거래금액) / 실제_거래금액 * 100

# 결과 출력
print(f"{'='*60}")
print(f"모델 검증: 신검단중앙역금강펜테리움센트럴파크 (2026년, 38건 평균)")
print(f"{'='*60}")
print(f"실제 거래금액 (평균): {실제_거래금액:,}만원 ({실제_거래금액/10000:.1f}억)")
print(f"예측 거래금액       : {예측_거래금액:,}만원 ({예측_거래금액/10000:.1f}억)")
print(f"오차               : {예측_거래금액 - 실제_거래금액:+,}만원")
print(f"오차율             : {오차율:.1f}%")

## 9. 실제 vs 예측 막대 그래프

In [ ]:
# 실제 거래금액과 예측 거래금액 비교 막대 그래프
fig, ax = plt.subplots(figsize=(8, 6))
labels = ['실제 거래금액', '예측 거래금액']
values = [실제_거래금액, 예측_거래금액]
colors = ['steelblue', 'orange']
ax.bar(labels, values, color=colors, width=0.4)
ax.set_ylabel('거래금액 (만원)')
ax.set_title(f'신검단중앙역금강펜테리움센트럴파크\n실제 vs 예측 거래금액 (오차율: {오차율:.1f}%)')
ax.set_ylim(0, max(values) * 1.2)
plt.tight_layout()
plt.show()